# Task 4 on Kaggle — baseline vs fast system, same session

Run on a **GPU T4** session (Settings → Accelerator → GPU T4 x1, Internet **on**).

Before running:
1. Upload the project folder `D:\inter_iit\TTS` (with `baseline/`, `data/`, `code/`, `metrics.py`) as a Kaggle **Dataset**.
   Leave out `predictions/` and `out/`; they are large and not needed.
2. Add it to this notebook (Add input). Set `DATASET` below to its folder name.
3. Add your Hugging Face token as a Kaggle **secret** named `HF_TOKEN` (Add-ons → Secrets).
4. **Run all.** It takes about 30–40 minutes on a T4.

What it does, in order:
1. Install the pinned stack (same fixes as the local setup).
2. Time the **baseline** on the 24 validation items (`baseline/run_baseline.py`).
3. Time the **fast system** (Task 3 idea + `nfe_step=FAST_NFE`) on the same 24 items, right after, same session.
4. Score the fast validation audio with `metrics.py`.
5. Synthesize the 40 evaluation items into `t4_fast/`.
6. Zip everything you need to download.

In [ ]:
DATASET = "tts-bootcamp"      # the folder name under /kaggle/input/
FAST_NFE = 16                 # flow-matching steps for the fast system (baseline uses 32)
SEED = 0

import os, shutil, subprocess, json, sys
SRC = f"/kaggle/input/{DATASET}"
KIT = "/kaggle/working/TTS"
if not os.path.exists(KIT):
    shutil.copytree(SRC, KIT, ignore=shutil.ignore_patterns("predictions", "out", "*.wav") )
    shutil.copytree(f"{SRC}/data/prompts", f"{KIT}/data/prompts", dirs_exist_ok=True)   # keep the prompt clips
os.chdir(KIT)
os.makedirs("out", exist_ok=True)

def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-4000:])
        raise RuntimeError(f"failed: {cmd}")
print(os.listdir(KIT))

## 1. Install
Kaggle already has torch with CUDA, so torch is not reinstalled. IndicF5 is installed from GitHub `main` without its own dependency list (its setup.py asks for an old numpy). The model weights stay pinned by `baseline/config.yaml`.

In [ ]:
sh('pip install -q --no-deps "git+https://github.com/ai4bharat/IndicF5.git@13f7c4d627cc10111aea8fe9c0039462cacacdc7"')
sh('pip install -q "numpy>=2.1,<2.3" transformers==4.49.0 accelerate soundfile librosa pyyaml pandas "speechbrain>=1.0" '
   '"huggingface-hub>=0.26,<1.0" cached_path click ema_pytorch hydra-core safetensors tomli torchdiffeq tqdm vocos '
   '"x_transformers>=1.31.14" pydub jieba pypinyin matplotlib transformers_stream_generator wandb "datasets<4"')
sh('python -c "import torch, transformers, numpy; print(torch.__version__, torch.cuda.is_available(), transformers.__version__, numpy.__version__)"')

If the numpy version changed above, use **Run → Restart & clear cell outputs**, then **Run all** again (once).

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

## 2. Apply the length fix and download the model
The first load patches the model's own code and stops on purpose. The second load works. Each runs in its own process, so no restart is needed.

In [ ]:
for attempt in (1, 2):
    r = subprocess.run('python -c "from baseline.tts import get_engine; get_engine()" && echo model ready',
                       shell=True, text=True, capture_output=True)
    print(attempt, (r.stdout + r.stderr)[-600:])

## 3. Time the baseline (24 validation items)
Model loading and one warm-up item are excluded from the time.

In [ ]:
sh(f"python baseline/run_baseline.py --items data/val_items.csv --out out/baseline_val --seed {SEED}")
base = json.load(open("out/baseline_val_timing.json"))
print("baseline seconds:", base["total_synth_s"], "on", base["gpu"])

## 4. Time the fast system, right after, in the same session

In [ ]:
sh(f"python code/run_system.py --mode idea --items data/val_items.csv --out out/val_fast --seed {SEED} --set nfe_step={FAST_NFE}")
fast = json.load(open("out/val_fast_timing.json"))
speedup = base["total_synth_s"] / fast["total_synth_s"]
print(f"fast seconds: {fast['total_synth_s']}   speed-up: {speedup:.2f}x")

## 5. Score the fast validation audio
Task 4 rule: its CER may be at most 1.2 × the Task 3 CER (0.0613 locally → limit about 0.074).

In [ ]:
sh("python metrics.py out/val_fast --json out/val_fast.json --csv out/val_fast.csv")
vf = json.load(open("out/val_fast.json"))
print("fast val CER", vf["cer"], " SIM", vf["sim"], " failed", vf["failed"])
print("limit check (<= 1.2 x 0.0613 = 0.0736):", "OK" if vf["cer"] <= 1.2 * 0.0613 else "TOO HIGH: raise FAST_NFE")

## 6. Synthesize the 40 evaluation items → `t4_fast/`

In [ ]:
sh(f"python code/run_system.py --mode idea --items data/eval_items.csv --out predictions/t4_fast --seed {SEED} --set nfe_step={FAST_NFE}")
print(len(os.listdir("predictions/t4_fast")), "files")

## 7. Summary and download
Download `task4_outputs.zip` from the Output panel. Put `t4_fast/` into `predictions/`, and both timing files into `code/`.

In [ ]:
summary = {"method_name": f"Task 3 system with nfe_step={FAST_NFE}", "settings": {"nfe_step": FAST_NFE},
           "val_cer": vf["cer"], "baseline_seconds": base["total_synth_s"], "fast_seconds": fast["total_synth_s"],
           "speedup": round(speedup, 3), "gpu": base["gpu"]}
json.dump(summary, open("out/task4_summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))
sh("cd /kaggle/working/TTS && zip -qr /kaggle/working/task4_outputs.zip predictions/t4_fast out/baseline_val_timing.json "
   "out/val_fast_timing.json out/val_fast.json out/val_fast.csv out/task4_summary.json")
print("ready: /kaggle/working/task4_outputs.zip")